In [24]:
import pandas as pd
import torch
import numpy as np
from models import Autoencoder
from utils import get_device, crear_datasets_proporcionales, estandarizar_columnas_no_binarias, separar_columnas_binarias

df = pd.read_csv("data/diabetes_012_health_indicators_BRFSS2015.csv")

# "Binarizamos" los datos, eliminando registros de pacientes con prediabetes
df = df[df["Diabetes_012"] != 1]
df["Diabetes_012"] = df["Diabetes_012"].replace(2, 1)

# Estandarizamos las columnas no binarias
df = estandarizar_columnas_no_binarias(df)
list_x_train, list_y_train, list_x_test, list_y_test, resumen_df = crear_datasets_proporcionales(df, "Diabetes_012")

device = get_device()

# Desempaquetamos los conjuntos train
x0_train, x1_train, x2_train, x3_train = list_x_train
y0_train, y1_train, y2_train, y3_train = list_y_train

# Desempaquetamos los conjuntos test
x0_test, x1_test, x2_test, x3_test = list_x_test
y0_test, y1_test, y2_test, y3_test = list_y_test

print(resumen_df)


Dispositivo usado: cuda (NVIDIA GeForce GTX 970)

   Proporción  Positivos  Negativos  Total  % Positivos  % Negativos
0        0.00          0      70692  70692          0.0        100.0
1        0.10       7069      63623  70692         10.0         90.0
2        0.25      17673      53019  70692         25.0         75.0
3        0.50      35346      35346  70692         50.0         50.0


In [25]:
idx_cols_binarias, idx_cols_no_binarias = separar_columnas_binarias(x0_train)
print(idx_cols_no_binarias)

[ 3 13 14 15 18 19 20]


In [26]:
modelo = Autoencoder.load(path="models/autoencoder_2025-11-26T19.32_lr=0.0001_set=0.pth", device=device)
#original = x0_train
original = x0_test

reconstruido = modelo.predict(x=original, device=device)

reconstruido[:, idx_cols_binarias] = np.round(np.abs(reconstruido[:, idx_cols_binarias]))

total = original.size
#aciertos = 100 * (np.sum(original == reconstruido)) / total
aciertos = np.mean((original - reconstruido)**2)

print(f"{aciertos}/{total} valores reconstruidos correctamente. Error de {aciertos:.2f}")
print("\nArray original:")
print(original[0])
print("\nArray reconstruido:")
print(reconstruido[0])

c:\entornos-gpu\anomaly-detection-with-autoencoder\models.py:103: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  modelo = torch.load(path, map_location=device)



Modelo cargado correctamente de 'models/autoencoder_2025-11-26T19.32_lr=0.0001_set=0.pth'
0.048396774437187595/3745497 valores reconstruidos correctamente. Error de 0.05

Array original:
[ 0.          0.          0.         -0.50633874  1.          0.
  0.          1.          0.          0.          0.          0.
  1.          0.46588173 -0.42814476 -0.48414917  0.          0.
 -0.3311125   0.96059071 -2.45224647]

Array reconstruido:
[ 0.          0.          1.         -0.39950317  1.          0.
  0.          1.          0.          1.          0.          1.
  0.          0.46586648 -0.4275306  -0.38981792  0.          0.
 -0.3432783   0.92643464 -2.6420949 ]


In [27]:
from models import HybridLossAutoencoder
modelo = HybridLossAutoencoder.load(path="models/autoencoder_2025-11-26T19.11_lr=0.0001_set=0.pth", device=device)

original = x0_test
reconstruido = modelo.predict(x=original, device=device)
reconstruido[:, idx_cols_binarias] = np.round(np.abs(reconstruido[:, idx_cols_binarias]))

aciertos = np.sum(original == reconstruido)
total = original.size

print(f"{aciertos}/{total} valores reconstruidos correctamente. Esto es un {100 * aciertos / total:.2f}%")
print("\nArray original:")
print(original[0])
print("\nArray reconstruido:")
print(reconstruido[0])


Modelo cargado correctamente de 'models/autoencoder_2025-11-26T19.11_lr=0.0001_set=0.pth'
1978929/3745497 valores reconstruidos correctamente. Esto es un 52.83%

Array original:
[ 0.          0.          0.         -0.50633874  1.          0.
  0.          1.          0.          0.          0.          0.
  1.          0.46588173 -0.42814476 -0.48414917  0.          0.
 -0.3311125   0.96059071 -2.45224647]

Array reconstruido:
[ 1.          0.          1.          0.2835574   1.          0.
  0.          1.          1.          1.          0.          1.
  0.          0.5482935  -0.28577256  0.0400501   0.          0.
  0.22293994 -0.8512959  -0.71477264]


In [28]:
from utils import evaluar_anomalias, obtener_metricas

resultados = evaluar_anomalias(modelo, x0_test, y0_test, device, 1)
print(resultados)

metricas = obtener_metricas(**resultados)

print("Matriz de confusión (valores):")
print(metricas["matriz_confusion"])
print("\nMatriz de confusión (%):")
print(metricas["matriz_confusion_pct"])

print(f"\nAccuracy : {metricas['accuracy']}")
print(f"Precisión: {metricas['precision']}")
print(f"Recall   : {metricas['recall']}")
print(f"F1-Score : {metricas['f1_score']}")

{'TP': 17186, 'FN': 18160, 'TN': 78247, 'FP': 64764}
Matriz de confusión (valores):
[[78247 64764]
 [18160 17186]]

Matriz de confusión (%):
[[43.87 36.31]
 [10.18  9.64]]

Accuracy : 0.5351
Precisión: 0.2097
Recall   : 0.4862
F1-Score : 0.293
